# Datamine DELETE Process Example

This notebook demonstrates how to configure and run the **`delete`** process wrapper in `dmstudio`.

## Process Description

## DELETE

Deletes the physical file associated with the symbolic &IN name from disk.

**Warning** : this process deletes physical Datamine files and cannot be undone. Be careful if running this command interactively, and particularly so if via a macro as part of batch process.

If the input file is a catalogue file then all files within the catalogue can be deleted, depending on the value of the optional parameter @**CONFIRM**. If the input is a catalogue file and @**CONFIRM** =0, the user is warned that operation is on catalogue file input and must confirm that deletion of all files in the catalogue is required, before this takes place. If the input is a catalogue file and @CONFIRM= 1, the user is again warned that operation is on catalogue file input, but in this case must confirm deletion of each file in the catalogue individually, before this takes place.

A catalogue file itself may be deleted by either overwriting it with another file before deletion; or by using the [DMEDIT](<dmedit.md>) process to change the name of the field from 'FILENAM to anything else.

The following **Output** window messages display on successful completion of files:

>>> nnnnnnnn RECORDS IN FILE filename
>>> FILE filename DELETED OK

### Input Files:

* **in** (Undefined):
  File to be deleted. If IN is a catalogue file, then all the files in the catalogue will
  be deleted if confirmed.
  Required=Yes

### Output Files:

### Fields:

### Parameters:

* **confirm**:

* **Options** (1;: If **IN** is a catalogue file, then a request for confirmation will be):
  issued for each file in the catalogue. If IN is a individual database file, then
  confirmation that the file is to be deleted is requested. Default: (0). >>> OPERATING ON

## A CATALOGUE FILE INPUT <<< >>> ARE YOU SURE YOU WISH TO DELETE ALL FILES <<< >>> PRESS

## <RETURN> TO CONTINUE (OR ! TO TERMINATE) >

  Range=0,1
  Values=0,1
  Default=0
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Initialize Sandbox
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Initialize active project sandbox using the shared test_sandbox project
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()
agent.initialize_sandbox(notebook_folder)

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('delete')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine database locally to this sandbox folder using a `t_` prefix.

In [ ]:
# Step 3: Copy VBOP datasets dynamically from Database to test_sandbox
files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

agent.initialize_sandbox(notebook_folder, files_to_copy=files_to_copy)

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute delete
print("Running delete...")
dm_cmd.delete(
    in_i='t_assays',  # required
    # confirm_p=0,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("delete execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = 't_delete_out.dmx'
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob("t_*.*")
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    if os.path.exists(f):
        try:
            os.remove(f)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = '__pycache__'
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")